**Import Libraries**

In [ ]:
import sys
import shutil
from glob import glob
import json
import math
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications import densenet
from keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from keras.preprocessing.image import ImageDataGenerator
from keras.utils.np_utils import to_categorical
from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score
import scipy
from tqdm import tqdm
import gc
from functools import partial
from sklearn import metrics
from collections import Counter
import json
import itertools

In [ ]:
import tensorflow as tf
print(tf.__version__)

**Problem Statement**

Breast cancer is the second most common cancer in women and men worldwide. 

Breast cancer starts when cells in the breast begin to grow out of control. These cells usually form a tumor that can often be seen on an x-ray or felt as a lump. The tumor is malignant (cancer) if the cells can grow into (invade) surrounding tissues or spread (metastasize) to distant areas of the body.

**The Challenge**

Build an algorithm to automatically identify whether a patient is suffering from breast cancer or not by looking at biopsy images.

**Data**

The dataset can be downloaded from https://web.inf.ufpr.br/vri/databases/breast-cancer-histopathological-database-breakhis/. 

This is a binary classification problem. The data is split into train/test using the below script:

In [ ]:
def create_folds_from_ds(dst_path='/kaggle/working'):
    """Creates a structure of directories containing images
        selected from BreaKHis_v1 dataset. 2,3,4,5
    """
    root_dir = '/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast'
    srcfiles = {'DC': '%s/malignant/SOB/ductal_carcinoma/%s/%sX/%s',
                'LC': '%s/malignant/SOB/lobular_carcinoma/%s/%sX/%s',
                'MC': '%s/malignant/SOB/mucinous_carcinoma/%s/%sX/%s',
                'PC': '%s/malignant/SOB/papillary_carcinoma/%s/%sX/%s',
                'A': '%s/benign/SOB/adenosis/%s/%sX/%s',
                'F': '%s/benign/SOB/fibroadenoma/%s/%sX/%s',
                'PT': '%s/benign/SOB/phyllodes_tumor/%s/%sX/%s',
                'TA': '%s/benign/SOB/tubular_adenoma/%s/%sX/%s'}

    
    # directory for nth-fold
    dst_dir = dst_path + '/Breast_Cancer_ Histopathological _Dataset'
    if not os.path.exists(dst_dir):
        os.mkdir(dst_dir)

    # image list
    db = open('/kaggle/input/image-files-loader/load_files.txt')
    benign_train = 0
    malignant_train = 0
    
    benign_test = 0
    malignant_test = 0
    
    for row in db.readlines():
        columns = row.split('|')
        imgname = columns[0]
        types = imgname.split('_')[1]
        if types == 'B':
            category = 'benign'
        elif types == 'M':
            category = 'malignant'
        mag = columns[1]  # 40, 100, 200, or 400
        grp = columns[3].strip()  # train or test
        
        dst_subdir = dst_dir + '/' + grp
        if not os.path.exists(dst_subdir):
            os.mkdir(dst_subdir)
        
        dst_subdir = dst_subdir + '/' + category
        if not os.path.exists(dst_subdir):
            os.mkdir(dst_subdir)
            
        tumor = imgname.split('-')[0].split('_')[-1]
        srcfile = srcfiles[tumor]
        
        s = imgname.split('-')
        sub = s[0] + '_' + s[1] + '-' + s[2]
        
        srcfile = srcfile % (root_dir, sub, mag, imgname)
        
        dstfile = dst_subdir + '/' + imgname
        
        if grp == 'train' and category == 'benign':
            if benign_train >= 250:
                continue
            else:
                benign_train +=1 
        
        if grp == 'train' and category == 'malignant':
            if malignant_train >= 250:
                continue
            else:
                malignant_train +=1
        
        if grp == 'test' and category == 'benign':
            if benign_test >= 250:
                continue
            else:
                benign_test +=1
        
        if grp == 'test' and category == 'malignant':
            if malignant_test >= 250:
                continue
            else:
                malignant_test +=1
            
        print ("Copying from [%s] to [%s]" % (srcfile, dstfile))
        shutil.copy(srcfile, dstfile)
    print('\n\n\t\t Train/Test Data Creation finished.\n')
    db.close()
    print ("\nProcess completed.")

In [ ]:
create_folds_from_ds()

**CNN Architecture**

THE Convolutional Neural Network (CNN) consists of the following:

1. **Convolution Layer**: 

The purpose of this layer is to receive a feature map. Usually, we start with low number of filters for low-level feature detection. The deeper we go into the CNN, the more filters we use to detect high-level features. Feature detection is based on ‘scanning’ the input with the filter of a given size and applying matrix computations in order to derive a feature map.

2. **Pooling Layer**:

The goal of this layer is to provide spatial variance, which simply means that the system will be capable of recognizing an object even when its appearance varies in some way. Pooling layer will perform a downsampling operation along the spatial dimensions (width, height), resulting in output such as [16x16x12] for pooling_size=(2, 2)

3. **Fully Connected Layer**:

In a fully connected layer, we flatten the output of the last convolution layer and connect every node of the current layer with the other nodes of the next layer. Neurons in a fully connected layer have full connections to all activations in the previous layer, as seen in regular Neural Networks and work in a similar way.

**Image Classification**

The complete image classification pipeline can be formalized as follows:

* Our input is a training dataset that consists of N images, each labeled with one of 2 different classes.

* Then, we use this training set to train a classifier to learn what every one of the classes looks like.

* In the end, we evaluate the quality of the classifier by asking it to predict labels for a new set of images that it has never seen before. We will then compare the true labels of these images to the ones predicted by the classifier.

**Load the Image from respective folders**

In [ ]:
def Dataset_loader(DIR, RESIZE, sigmaX=10):
    IMG = []
    read = lambda imname: np.asarray(Image.open(imname).convert("RGB"))
    for IMAGE_NAME in tqdm(os.listdir(DIR)):
        PATH = os.path.join(DIR,IMAGE_NAME)
        _, ftype = os.path.splitext(PATH)
        if ftype == ".png":
            img = read(PATH)
           
            img = cv2.resize(img, (RESIZE,RESIZE))
           
            IMG.append(np.array(img))
    return IMG

benign_train = np.array(Dataset_loader('/kaggle/working/Breast_Cancer_ Histopathological _Dataset/train/benign/',224))
malign_train = np.array(Dataset_loader('/kaggle/working/Breast_Cancer_ Histopathological _Dataset/train/malignant/',224))
benign_test = np.array(Dataset_loader('/kaggle/working/Breast_Cancer_ Histopathological _Dataset/test/benign/',224))
malign_test = np.array(Dataset_loader('/kaggle/working/Breast_Cancer_ Histopathological _Dataset/test/malignant/',224))

After loading the images, create a numpy array of zeroes for labeling benign images and similarly a numpy array of ones for labeling malignant images. 

Also shuffled the dataset and converted the labels into categorical format.

In [ ]:
benign_train_label = np.zeros(len(benign_train))
malign_train_label = np.ones(len(malign_train))
benign_test_label = np.zeros(len(benign_test))
malign_test_label = np.ones(len(malign_test))

X_train = np.concatenate((benign_train, malign_train), axis = 0)
Y_train = np.concatenate((benign_train_label, malign_train_label), axis = 0)
X_test = np.concatenate((benign_test, malign_test), axis = 0)
Y_test = np.concatenate((benign_test_label, malign_test_label), axis = 0)

s = np.arange(X_train.shape[0])
print('Number of training images', X_train.shape[0])

np.random.shuffle(s)
X_train = X_train[s]
Y_train = Y_train[s]

s = np.arange(X_test.shape[0])
print('Number of testing images', X_test.shape[0])
np.random.shuffle(s)
X_test = X_test[s]
Y_test = Y_test[s]

Y_train = to_categorical(Y_train, num_classes= 2)
Y_test = to_categorical(Y_test, num_classes= 2)

**Train-Test Split**

Split the data-set into two sets — train and test sets with 80% and 20% images respectively

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    X_train, Y_train, 
    test_size=0.2, 
    random_state=11
)

Let’s see some sample benign and malignant images.

In [ ]:
w=60
h=40
fig=plt.figure(figsize=(15, 15))
columns = 4
rows = 3

for i in range(1, columns*rows +1):
    ax = fig.add_subplot(rows, columns, i)
    if np.argmax(Y_train[i]) == 0:
        ax.title.set_text('Benign')
    else:
        ax.title.set_text('Malignant')
    plt.imshow(x_train[i], interpolation='nearest')
plt.show()

**Batch Size**

Batch size is one of the most important hyperparameters to tune in deep learning. A larger batch size trains models as it allows computational speedups from the parallelism of GPUs. However, it is well known that too large of a batch size will lead to poor generalization. On the one extreme, using a batch equal to the entire dataset guarantees convergence to the global optima of the objective function. However this is at the cost of slower convergence to that optima. On the other hand, using smaller batch sizes have been shown to have faster convergence to good results. This is intuitively explained by the fact that smaller batch sizes allow the model to start learning before having to see all the data. The downside of using a smaller batch size is that the model is not guaranteed to converge to the global optima.Therefore it is often advised that one starts at a small batch size reaping the benefits of faster training dynamics and steadily grows the batch size through training.

In [ ]:
BATCH_SIZE = 16

**Data Augmentation**

The practice of data augmentation is an effective way to increase the size of the training set. Augmenting the training examples allow the network to see more diversified, but still representative data points during training.

In [ ]:
train_generator = ImageDataGenerator(
        zoom_range=2,  # set range for random zoom
        rotation_range = 90,
        horizontal_flip=True,  # randomly flip images
        vertical_flip=True,  # randomly flip images
    )

**Building the Model**

This can be described in the following 3 steps:

* Use ResNet101V2 as the pre trained weights which is already trained in the Imagenet competition. The learning rate was chosen to be 0.0001.

* On top of it, use a globalaveragepooling layer followed by 50% dropouts to reduce over-fitting.

* Use batch normalization and a dense layer with 2 neurons for 2 output classes ie benign and malignant with softmax as the activation function.

* Use Adam as the optimizer and binary-cross-entropy as the loss function.

In [ ]:
from keras.applications.resnet_v2 import ResNet101V2
from keras import layers

def build_model(backbone, lr=1e-4):
    model = Sequential()
    model.add(backbone)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dropout(0.5))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(2, activation='softmax'))
    
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(lr=lr),
        metrics=['accuracy']
    )
    return model

resnet = ResNet101V2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

model = build_model(resnet ,lr = 1e-4)
model.summary()

**Model Callbacks**

Before training the model, it is useful to define one or more callbacks. Pretty handy one, are: ModelCheckpoint and ReduceLROnPlateau.

* **ModelCheckpoint**: When training requires a lot of time to achieve a good result, often many iterations are required. In this case, it is better to save a copy of the best performing model only when an epoch that improves the metrics ends.

* **ReduceLROnPlateau**: Reduce learning rate when a metric has stopped improving. Models often benefit from reducing the learning rate by a factor of 2–10 once learning stagnates. This callback monitors a quantity and if no improvement is seen for a ‘patience’ number of epochs, the learning rate is reduced.

**Model Training**

Model was trained for 5 epochs

In [ ]:
learn_control = ReduceLROnPlateau(monitor='val_acc', patience=5,
                                  verbose=1,factor=0.2, min_lr=1e-7)

filepath="/kaggle/working/weights_best.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='val_acc', verbose=1, save_best_only=True, mode='max')

#Train Model
history = model.fit_generator(
    train_generator.flow(x_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=x_train.shape[0] / BATCH_SIZE,
    epochs=5,
    validation_data=(x_val, y_val),
    callbacks=[learn_control, checkpoint])

**Model Loss/Accuracy curve**

In [ ]:
history_df = pd.DataFrame(history.history)
history_df

In [ ]:
history_df[['loss', 'val_loss']].plot()

history_df[['accuracy', 'val_accuracy']].plot()

**Prediction**

In [ ]:
Y_pred = model.predict(x_val)

In [ ]:
print('Model Accuracy score', accuracy_score(np.argmax(y_val, axis=1), np.argmax(Y_pred, axis=1)))

**Performance Metrics**

The most common metric for evaluating model performance is the accurcacy. However, when only 2% of your dataset is of one class (malignant) and 98% some other class (benign), misclassification scores don’t really make sense. You can be 98% accurate and still catch none of the malignant cases which could make a terrible classifier.

**Precision, Recall and F1-Score**

For a better look at misclassification, we often use the following metric to get a better idea of true positives (TP), true negatives (TN), false positive (FP) and false negative (FN).

**Precision** is the ratio of correctly predicted positive observations to the total predicted positive observations.

**Recall** is the ratio of correctly predicted positive observations to all the observations in actual class.

**F1-Score** is the harmonic mean of Precision and Recall.

**Confusion Matrix**

Confusion Matrix is a very important metric when analyzing misclassification. Each row of the matrix represents the instances in a predicted class while each column represents the instances in an actual class. The diagonals represent the classes that have been correctly classified. This helps as we not only know which classes are being misclassified but also what they are being misclassified as.

In [ ]:
tta_steps = 10
predictions = []

for i in tqdm(range(tta_steps)):
    preds = model.predict_generator(train_generator.flow(X_test, batch_size=BATCH_SIZE, shuffle=False),
                                    steps = len(X_test)/BATCH_SIZE)
    
    predictions.append(preds)
    gc.collect()
    
Y_pred_tta = np.mean(predictions, axis=0)

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=55)
    plt.yticks(tick_marks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

cm = confusion_matrix(np.argmax(Y_test, axis=1), np.argmax(Y_pred_tta, axis=1))

cm_plot_label =['benign', 'malignant']
plot_confusion_matrix(cm, cm_plot_label, title ='Confusion Metrix for Breast Cancer')

In [ ]:
from sklearn.metrics import classification_report

classification_report( np.argmax(Y_test, axis=1), np.argmax(Y_pred_tta, axis=1))

**ROC Curves**

The 45 degree line is the random line, where the Area Under the Curve or AUC is 0.5 . The further the curve from this line, the higher the AUC and better the model. The highest a model can get is an AUC of 1, where the curve forms a right angled triangle. The ROC curve can also help debug a model. For example, if the bottom left corner of the curve is closer to the random line, it implies that the model is misclassifying at Y=0. Whereas, if it is random on the top right, it implies the errors are occurring at Y=1.

In [ ]:
from sklearn.metrics import roc_auc_score, auc
from sklearn.metrics import roc_curve
roc_log = roc_auc_score(np.argmax(Y_test, axis=1), np.argmax(Y_pred_tta, axis=1))
false_positive_rate, true_positive_rate, threshold = roc_curve(np.argmax(Y_test, axis=1), np.argmax(Y_pred_tta, axis=1))
area_under_curve = auc(false_positive_rate, true_positive_rate)

plt.plot([0, 1], [0, 1], 'r--')
plt.plot(false_positive_rate, true_positive_rate, label='AUC = {:.3f}'.format(area_under_curve))
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend(loc='best')
plt.show()
plt.close()

**Conclusions**

It is remarkable to see the success of deep learning in such varied real world problems. In this blog, we have demonstrated how to classify benign and malignant breast cancer from a collection of microscopic images using convolutional neural networks and transfer learning.